# Capstone: Audit a Classifier's Fairness — and Hit the Wall

Capstone notebook for the [Fairness, Formally](https://josephausterweil.github.io/probintro/ethics/fairness-formalisms/) chapter (Part IX).

**The experiment.** Three notions of fairness — demographic parity, equalized odds, and calibration — are each an equality between **conditional probabilities**, and when two groups have different **base rates** they cannot all hold at once. You will turn the three definitions into an auditing tool, then reproduce that impossibility on a bigger, messier, COMPAS-like dataset. You will:

- **(A)** implement the three fairness criteria as functions on a **confusion matrix**;
- **(B)** on a synthetic two-group dataset with **unequal base rates** and a **calibrated** score, pick a threshold, confirm calibration, and measure the **equalized-odds violation** — the two groups' false-positive rates diverging;
- **(C, stretch)** sweep group-specific thresholds and **plot the fairness frontier**: every point that closes the FPR gap opens the PPV gap.

**Anchors.** Kleinberg, Mullainathan & Raghavan (2016) and Chouldechova (2017) are the impossibility results you are reproducing. The ProPublica/Northpointe **COMPAS** exchange (2016) is the real case the synthetic data mimics: ProPublica measured an error-rate (equalized-odds) gap, Northpointe measured calibration (predictive parity), and both were correct — the base rates differed, so the identity forbids satisfying both. Here you set the data up yourself, so you know *exactly* why the wall appears.


## 0. Setup (complete — run me)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
print("numpy", np.__version__, "ready")


def confusion(y_true, y_pred):
    """Confusion matrix [[TN, FP], [FN, TP]] (rows = truth Y, cols = prediction Yhat)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    return np.array([[TN, FP], [FN, TP]])


def show(M, title=""):
    (TN, FP), (FN, TP) = M
    print(f"{title}  TN={TN} FP={FP} FN={FN} TP={TP}")

## A. The three criteria as code — TODO

Each fairness criterion is a conditional probability you read off a confusion matrix `M = [[TN, FP], [FN, TP]]`:

- **Demographic parity** compares $P(\hat{Y}=1 \mid G)$ — the *approval rate* — across groups.
- **Equalized odds** compares the **TPR** $=P(\hat{Y}=1 \mid Y=1)$ and **FPR** $=P(\hat{Y}=1 \mid Y=0)$ across groups.
- **Calibration at the threshold** shows up as equal **PPV** $=P(Y=1 \mid \hat{Y}=1)$ (predictive parity).

Fill in the three functions, then uncomment the smoke test to check them against the chapter's hand-worked numbers.

In [ ]:
def approval_rate(M):
    """Demographic-parity quantity: P(Yhat=1) for one group. Equal across groups => parity."""
    # TODO: return (FP + TP) / M.sum()
    raise NotImplementedError


def tpr_fpr(M):
    """Equalized-odds quantities: (TPR, FPR) = (P(Yhat=1|Y=1), P(Yhat=1|Y=0))."""
    # TODO:
    #   (TN, FP), (FN, TP) = M
    #   return TP / (TP + FN), FP / (FP + TN)
    raise NotImplementedError


def ppv(M):
    """Calibration-at-threshold quantity: PPV = P(Y=1 | Yhat=1) (predictive parity)."""
    # TODO:
    #   (TN, FP), (FN, TP) = M
    #   return TP / (TP + FP)
    raise NotImplementedError


# Smoke test against the chapter (uncomment once all three are implemented):
# M_sakura = np.array([[ 75, 100], [ 25, 300]])   # [[TN, FP], [FN, TP]]
# M_kaede  = np.array([[300,  25], [100,  75]])
# print("approval rates:", approval_rate(M_sakura), approval_rate(M_kaede))   # 0.80, 0.20
# print("Sakura TPR,FPR:", tpr_fpr(M_sakura))     # (0.923, 0.571)
# print("Kaede  TPR,FPR:", tpr_fpr(M_kaede))      # (0.429, 0.077)
# print("PPV both:", ppv(M_sakura), ppv(M_kaede)) # 0.75, 0.75

## B. The impossibility on a COMPAS-like dataset

The dataset below has two groups with **unequal base rates**, and a score that is **calibrated across groups by construction**: a given score value means the same success probability in both groups (because each outcome is drawn as `Bernoulli(score)`). This cell is complete — run it and read off the two base rates.

In [ ]:
N = 4000
group = rng.integers(0, 2, size=N)                 # 0 = group A, 1 = group B

# Latent "true success probability" per person = the calibrated score, in (0, 1).
# Group A is drawn toward HIGH scores, group B toward LOW scores, so their base rates differ;
# but a given score value means the same success probability in both groups.
score = np.where(group == 0,
                 rng.beta(2.5, 1.5, size=N),        # group A: mass toward high scores
                 rng.beta(1.5, 2.5, size=N))        # group B: mass toward low scores
y = (rng.random(N) < score).astype(int)            # outcome ~ Bernoulli(score): calibrated

for g, name in ((0, "A"), (1, "B")):
    m = group == g
    print(f"group {name}: n={int(m.sum()):4d}  base_rate P(Y=1)={y[m].mean():.3f}  "
          f"mean_score={score[m].mean():.3f}")

### B (audit) — TODO

Threshold the calibrated score, then show the wall: calibration holds, yet the false-positive rates split.

In [ ]:
# TODO (B): audit the thresholded classifier.
#   1. Pick a threshold t and predict  yhat = (score >= t).
#   2. Per group, build the confusion matrix with confusion(y[mask], yhat[mask]).
#   3. CALIBRATION check: within a thin score band (e.g. score in [t, t+0.1]), P(Y=1) is
#      ~equal across groups -- the score means the same thing in both.
#   4. EQUALIZED-ODDS check: print both groups' FPR from tpr_fpr(...). They DIVERGE.
#
# t = 0.5
# yhat = (score >= t).astype(int)
# for g, name in ((0, "A"), (1, "B")):
#     m = group == g
#     M = confusion(y[m], yhat[m])
#     TPR, FPR = tpr_fpr(M)
#     print(f"group {name}: PPV={ppv(M):.3f}  TPR={TPR:.3f}  FPR={FPR:.3f}")
#
# Question to answer in a sentence: PPV is ~equal (calibration), but the FPRs are not.
# Using Chouldechova's identity FPR = (p/(1-p)) * ((1-PPV)/PPV) * (1-FNR), explain why NO
# single global threshold t can make both the FPRs and the PPVs match while base rates differ.

## C. Stretch: the fairness frontier — TODO

Give each group its *own* threshold and watch the trade-off become a curve you cannot cross.

In [ ]:
# TODO (C, stretch): plot the fairness frontier.
#   Sweep group-specific thresholds t_A, t_B over a grid. For each (t_A, t_B) pair:
#       gap_odds = |FPR_A - FPR_B|      # equalized-odds violation
#       gap_ppv  = |PPV_A - PPV_B|      # calibration / predictive-parity violation
#   Scatter gap_odds (x) vs gap_ppv (y). The empty lower-left corner (both near 0) is exactly
#   the region the impossibility forbids: closing one gap opens the other.
#
# ts = np.linspace(0.2, 0.8, 25)
# mA, mB = group == 0, group == 1
# pts = []
# for tA in ts:
#     for tB in ts:
#         MA = confusion(y[mA], (score[mA] >= tA).astype(int))
#         MB = confusion(y[mB], (score[mB] >= tB).astype(int))
#         gap_odds = abs(tpr_fpr(MA)[1] - tpr_fpr(MB)[1])
#         gap_ppv  = abs(ppv(MA) - ppv(MB))
#         pts.append((gap_odds, gap_ppv))
# xs, ys = zip(*pts)
# plt.scatter(xs, ys, s=6, alpha=0.5)
# plt.xlabel("|FPR_A - FPR_B|  (equalized-odds gap)")
# plt.ylabel("|PPV_A - PPV_B|  (calibration gap)")
# plt.title("The fairness frontier: the lower-left corner stays empty")
# plt.show()

---

## References

- Angwin, J., Larson, J., Mattu, S., & Kirchner, L. (2016). Machine bias. *ProPublica*.
- Chouldechova, A. (2017). Fair prediction with disparate impact: A study of bias in recidivism prediction instruments. *Big Data, 5*(2), 153–163. <https://doi.org/10.1089/big.2016.0047>
- Dieterich, W., Mendoza, C., & Brennan, T. (2016). *COMPAS Risk Scales: Demonstrating Accuracy Equity and Predictive Parity*. Northpointe Inc.
- Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning. *NeurIPS 29*. <https://arxiv.org/abs/1610.02413>
- Kleinberg, J., Mullainathan, S., & Raghavan, M. (2016). Inherent trade-offs in the fair determination of risk scores. *ITCS 2017*. <https://arxiv.org/abs/1609.05807>

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*
